# Compile a Hazus earthquake inventory of a city

This notebook builds a building inventory of Berkeley, California, in the form a Hazus earthquake damage and loss assessment reads. BRAILS++ takes the name of the city and its seismic zone, and gathers every other input from public sources:

- the city boundary and the census block groups, from the Census Bureau's TIGER/Line files;
- the building footprints, from Overture Maps;
- the building records of the National Structure Inventory (NSI);
- the housing tables of the American Community Survey (ACS), from the Census Bureau;
- the locations of mobile home parks, from an archive of the Homeland Infrastructure Foundation-Level Data (HIFLD).

Some values, such as the year a building was built, are not recorded for each building in these sources, and BRAILS++ draws them at random from census counts. The inventory therefore holds ten possible worlds: ten versions of the inventory that differ only in the drawn values.

In [ ]:
# To compile another city, change these two values. The section "Compile the
# inventory" below says where the seismic zone of a city is found.
CITY = 'Berkeley, CA'
SEISMIC_ZONE = '4'

## Store the Census API key

BRAILS++ reads the ACS tables through the Census Bureau's Data API, which needs a free Census API key. Request one at https://api.census.gov/data/key_signup.html. The Census Bureau sends the key by email with a link, and the key works only after you open that link.

The cell below looks for a stored key. If it finds none, it asks for the key without showing what you type and writes it to the file `.brails/census_api_key` in your home folder, where later runs find it. On macOS and Linux only you can read that file. On Windows the file takes the permissions of your home folder. You can instead set the environment variable `CENSUS_API_KEY` before you start Jupyter. BRAILS++ reads the variable before the file. To replace a stored key, for example one typed wrong, run `save_census_api_key()` in a cell of its own, which writes over the file.

Do not type the key into a cell of this notebook: a saved notebook keeps its cells and their output.

In [ ]:
from brails.utils.census_api_key import find_census_api_key, save_census_api_key

try:
    find_census_api_key()
    print('A Census API key is stored.')
except LookupError:
    save_census_api_key()
    print('The Census API key is now stored.')

## Compile the inventory

`compile_place` takes the name of the city with its state. It first asks the Census Data API whether it accepts the key. It then downloads the city boundary, the block groups, the footprints, the NSI records, the park locations and the ACS tables, and compiles the ten possible worlds. It reads the city and the census block groups around it, so the counts it prints while reading are larger than the city's, and it drops footprints under 450 square feet as it reads them. The compilation takes most of the time, and the footprints most of the memory: BRAILS++ reads them from the Overture Maps files in the cloud, which needs about 4 GB. For a city of Berkeley's size the cell runs for many minutes, and a smaller place, such as a small town, gives a quicker first run.

The compiler needs the seismic zone of the 1997 Uniform Building Code, from which it assigns the Hazus design levels. Berkeley is in zone 4. The zone of another city is found on the code's seismic zone map, Figure 16-2, and is one of `'4'`, `'3'`, `'2B'`, `'2A'`, `'1'` and `'0'`. In California the compiler takes only `'3'` or `'4'`, and it does not check the zone against the city. The compiler takes places in Alaska, Arizona, California, Colorado, Hawaii, Idaho, Montana, Nevada, New Mexico, Oregon, Utah, Washington and Wyoming, and refuses a place elsewhere with a message. The seed fixes the random draws, so a run with the same seed and the same data gives the same inventory. Any integer serves as the seed. A later run can differ where a source has changed since.

The lines the cell prints on units and on the Overture release are notices and need no action. If a download fails, the cell stops with an error, and the footprint read in particular has no retry. Run the cell again, and every download starts over. To keep the footprints and NSI records once they are read, so that a later failure does not lose them, call `CompilerInputs.from_place` (from `brails.compilers`) and then the compiler's `compile`, as the docstring of `compile_place` describes.

In [ ]:
from brails.compilers import HazusEQInventoryCompiler

compiler = HazusEQInventoryCompiler(seed=20261111, seismic_zone=SEISMIC_ZONE)
result = compiler.compile_place(CITY)

## Write the inventory

`write` saves the inventory in the folder of this notebook, in four files named after the city, which for Berkeley are:

- `berkeley.geojson`: the buildings, each with its footprint. A property whose value differs between the possible worlds holds a list of ten values, one for each world.
- `berkeley_report_summary.json`: a summary of the run, with its settings, the buildings left out and why, and notices.
- `berkeley_report_exclusions.parquet`: each building left out, with the reason, in each possible world.
- `berkeley_manifest.json`: a checksum of each intermediate result. Two runs with the same checksums computed the same values.

On Windows, close these files in other programs, such as QGIS, before you run the cell again: a file open in another program cannot be replaced.

In [ ]:
inventory_file = CITY.split(',')[0].lower().replace(' ', '_') + '.geojson'
result.write(inventory_file)

## The result

Of the footprints it keeps, the compiler drops small ones, such as garages and sheds, to which it gives no housing units. It excludes a building from a world in which it cannot give it every value, and `write` leaves out a building excluded from any world. The counts below are of the city alone, and the buildings written are its footprints less those dropped or excluded:

In [ ]:
summary = result.report.summary()
excluded = summary['assets_excluded']
print(
    summary['boundary']['inside'],
    'footprints inside the city,',
    excluded['dropped'],
    'small ones dropped,',
    excluded['every'],
    'buildings excluded from every world,',
    excluded['some'],
    'from some worlds',
)

The cell below reads the written file back and takes world 0, the first possible world. The number of its buildings by structure type and design level:

In [ ]:
import geopandas as gpd

from brails.types import AssetInventory

inventory = AssetInventory()
inventory.read_from_geojson(inventory_file)
world_0 = inventory.get_world_realization(0)
buildings = gpd.GeoDataFrame.from_features(
    world_0.get_geojson()['features'], crs='EPSG:4326'
)
counts = buildings.groupby(['StructureType', 'DesignLevel']).size()
counts.unstack(fill_value=0).reindex(
    columns=['Pre-Code', 'Low-Code', 'Moderate-Code', 'High-Code'], fill_value=0
)

A map of the year built of each building in world 0. The basemap is Esri's World Light Gray Canvas, and the footprints are Overture Maps data under the Open Database License (ODbL):

In [ ]:
import contextily

ax = buildings.to_crs(epsg=3857).plot(
    column='YearBuilt',
    cmap='viridis',
    legend=True,
    legend_kwds={'label': 'Year built', 'shrink': 0.6},
    figsize=(10, 10),
)
contextily.add_basemap(ax, source=contextily.providers.Esri.WorldGrayCanvas)
ax.set_title('Year built, world 0\nFootprints: Overture Maps (ODbL)')
ax.set_axis_off()